<a href="https://colab.research.google.com/github/philhanna/word-scorer/blob/main/ScoreWords.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

First, let's mount your Google Drive. You'll be prompted to authorize access.

In [17]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
import pandas as pd
import numpy as np

Download the clues

In [19]:
URL = "https://xd.saul.pw/xd-clues.zip"

In [24]:
import requests
import zipfile
import io
import pandas as pd

# Define the URL and the target file within the zip
zip_url = URL
file_in_zip = 'xd/clues.tsv'

# Download the zip file content
response = requests.get(zip_url)
response.raise_for_status() # Raise an exception for bad status codes

# Use BytesIO to treat the content as a file-like object
zip_file_object = io.BytesIO(response.content)

# Open the zip file
with zipfile.ZipFile(zip_file_object, 'r') as zf:
    # Read the specific file into a string
    with zf.open(file_in_zip) as f:
        tsv_content = f.read().decode('utf-8')

# Read the TSV content into a pandas DataFrame, handling bad lines by warning
df = pd.read_csv(io.StringIO(tsv_content), sep='\t', on_bad_lines='warn')

print(f"Successfully loaded '{file_in_zip}' into a DataFrame with {len(df)} rows and {len(df.columns)} columns.")
print(df.head())

/tmp/ipykernel_1537/1120636094.py:24: ParserWarning: Skipping line 2898192: expected 4 fields, saw 5

  df = pd.read_csv(io.StringIO(tsv_content), sep='\t', on_bad_lines='warn')


Successfully loaded 'xd/clues.tsv' into a DataFrame with 7997956 rows and 4 columns.
  pubid  year  answer                                               clue
0   7xw  2021    1138  THX ___ (George Lucas film referenced in many ...
1   7xw  2021  125243  Prime whose first three digits are p^q and who...
2   7xw  2021  158719  Multiple of XI (except not in Roman numerals; ...
3   7xw  2021    1812             Overture whose use of cannons is canon
4   7xw  2021   1IRON  Not even God can hit the ___ --Lee Trevino, af...


In [25]:
df = df.groupby(['answer']).size().reset_index(name='count')
df = df.rename(columns={'answer': 'word'})

Reject any words that are not simple ASCII letters

In [29]:
import re

# Filter df to keep only words that are entirely uppercase letters A-Z
df = df[df['word'].apply(lambda x: isinstance(x, str) and bool(re.fullmatch(r'[A-Z]+', x)))].copy()

print("df after filtering for entirely uppercase words:")
print(df.head())

df after filtering for entirely uppercase words:
       word  count     score
429      AA      3  0.108172
430     AAA   1558  0.573656
431    AAAA    106  0.364618
432   AAAAA      4  0.125583
433  AAAAAA      1  0.054086


## Add Log-Scaled Frequency (The "Diminishing Returns" Score)

In language, a few words are used constantly, while most words are used rarely (Zipf's Law). To prevent your scores from dropping to near-zero too fast, you can use a log scale. This assumes that once a publisher uses a word a handful of times, it's "acceptable," and using it 100 more times doesn't make it drastically more acceptable.$$Score = \frac{\ln(Count_{word} + 1)}{\ln(Count_{max} + 1)}$$

In [30]:
df['score'] = np.log(df['count'] + 1) / np.log(len(df))

print("DataFrame df created successfully with the counts, column renamed, and score calculated:")
print(df.head())

DataFrame df created successfully with the counts, column renamed, and score calculated:
       word  count     score
429      AA      3  0.108172
430     AAA   1558  0.573656
431    AAAA    106  0.364618
432   AAAAA      4  0.125583
433  AAAAAA      1  0.054086


In [31]:
df.to_csv("scored_words.csv", index=False)
print("df saved to scored_words.csv")

df saved to scored_words.csv
